# watchin-u: 2. Train and evaluate (Colab GPU)

Fine-tunes the three models on the datasets built by notebook 1 (`DRIVE/datasets/*.zip`),
evaluates them on the **frozen gold test split**, and writes one summary to paste back into
the Claude chat.

| Model | Base | Job |
|---|---|---|
| detect | `yolo26s.pt` (COCO) | coil + person boxes (counts, slots, occlusion) |
| stage  | `yolo26n-cls.pt`   | no_truck / wire_rod / other_load / covering / covered (the light trigger) |
| layout | `yolo26s-cls.pt`   | linear / criss_cross / mixed / not_visible |

Runtime -> Change runtime type -> **GPU** (T4 works; L4/A100 are faster). Runs are written
straight to Drive, so a disconnect loses nothing: use the resume cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup: Drive, code, install

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---- settings ------------------------------------------------------------------
DRIVE = "/content/drive/MyDrive/watchinu"  # project folder in your Drive (created if missing)
REPO = "nav0225/watchin-u"
BRANCH = "ccr-09c28bb1-bjcddo"
# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left bar, read-only
# token). Or upload the code as  DRIVE/code/watchin-u.zip  and it is used instead.

import os, pathlib, shutil, subprocess, zipfile

pathlib.Path(DRIVE).mkdir(parents=True, exist_ok=True)
CODE = pathlib.Path("/content/watchin-u")
if CODE.exists():
    shutil.rmtree(CODE)
code_zip = pathlib.Path(DRIVE) / "code" / "watchin-u.zip"
if code_zip.exists():
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/content/_code")
    root = next(p.parent for p in pathlib.Path("/content/_code").rglob("pyproject.toml"))
    shutil.move(str(root), CODE)
    print("code from", code_zip)
else:
    token = ""
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone failed - private repo? add GITHUB_TOKEN or the code zip\n" + msg)
print(subprocess.run(["git", "-C", str(CODE), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e "/content/watchin-u[ml]"
%cd /content/watchin-u
!watchinu --help | head -5

In [ ]:
RUN_TAG = "v1"  # bump for every iteration: v2, v3, ...
TRAIN = {"detect": True, "stage": True, "layout": True}
EPOCHS = {"detect": 100, "stage": 30, "layout": 50}
RUNS = f"{DRIVE}/runs"

# Datasets: copy from Drive to the local disk (training straight from Drive is slow).
DS = pathlib.Path("/content/ds")
for name in TRAIN:
    z = pathlib.Path(DRIVE) / "datasets" / f"{name}.zip"
    if not z.exists():
        print(f"missing {z} - skipping {name}")
        TRAIN[name] = False
        continue
    shutil.rmtree(DS / name, ignore_errors=True)
    shutil.unpack_archive(str(z), str(DS / name))
    card = DS / name / "dataset_card.json"
    print(name, open(card).read() if card.exists() else "(no dataset card)")

In [ ]:
def train(key, data, model, imgsz, hyp, batch=-1):
    cmd = ["watchinu", "train", "--data", str(data), "--model", model, "--imgsz", str(imgsz),
           "--epochs", str(EPOCHS[key]), "--batch", str(batch), "--device", "0",
           "--hyp", hyp, "--project", RUNS, "--name", f"{key}-{RUN_TAG}"]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


if TRAIN["detect"]:
    train("detect", DS / "detect" / "data.yaml", "yolo26s.pt", 1280, "configs/train/detector.yaml")

In [ ]:
if TRAIN["stage"]:
    train("stage", DS / "stage", "yolo26n-cls.pt", 224, "configs/train/classifier.yaml", batch=64)
if TRAIN["layout"]:
    train("layout", DS / "layout", "yolo26s-cls.pt", 384, "configs/train/classifier.yaml", batch=32)

### Resume after a disconnect
Uncomment and point at the interrupted run's `last.pt`.

In [ ]:
# !yolo train resume model={RUNS}/detect-{RUN_TAG}/weights/last.pt

## Evaluate on the gold test split and write the summary

In [ ]:
import glob


def latest_run(key):
    runs = sorted(glob.glob(f"{RUNS}/{key}-{RUN_TAG}*"), key=os.path.getmtime)
    return pathlib.Path(runs[-1]) if runs else None


def pick_split(root, detect):
    base = root / "images" if detect else root
    return "test" if (base / "test").exists() else "val"


sections = [f"# watchin-u results {RUN_TAG}\n"]
for key, task, data, imgsz in [
    ("detect", "detect", DS / "detect" / "data.yaml", 1280),
    ("stage", "classify", DS / "stage", 224),
    ("layout", "classify", DS / "layout", 384),
]:
    run = latest_run(key)
    if run is None or not (run / "weights" / "best.pt").exists():
        sections.append(f"## {key}: not trained\n")
        continue
    split = pick_split(DS / key, task == "detect")
    out = run / f"report_{split}"
    subprocess.run(["watchinu", "evaluate-model", "--task", task, "--weights",
                    str(run / "weights" / "best.pt"), "--data", str(data), "--split", split,
                    "--imgsz", str(imgsz), "--device", "0", "--out", str(out)], check=True)
    sections.append(out.with_suffix(".md").read_text())

summary = "\n".join(sections)
path = pathlib.Path(RUNS) / f"summary-{RUN_TAG}.md"
path.write_text(summary)
print(summary)
print("\nsaved:", path, "<- paste this into the Claude chat")

## Also useful to share
* `runs/detect-<tag>/confusion_matrix.png`, `results.png`, `val_batch0_pred.jpg` (image uploads)
* the `summary-<tag>.md` above

## Use the weights
Copy `best.pt` of each run to the server and set them in `configs/camera01.yaml`:
```yaml
models:
  stage_classifier: models/stage-v1.pt
  layout_classifier: models/layout-v1.pt
  detector: models/detect-v1.pt
```
Optional ONNX export for CPU-only servers (Ultralytics loads `.onnx` the same way):

In [ ]:
# run = latest_run("detect")
# !yolo export model={run}/weights/best.pt format=onnx imgsz=1280